# Heart Disease DNN Classification

A deep neural network workflow for binary heart disease prediction using tabular clinical attributes, preprocessing, hyperparameter experiments, and diagnostic evaluation.

## What this notebook demonstrates

- Load and inspect the heart disease dataset.
- Encode categorical features and scale numerical attributes.
- Train multiple DNN configurations with early stopping.
- Compare model variants and select the best-performing setup.
- Evaluate using classification report, confusion matrix, ROC curve, and AUC.



# Project #1 – Part 1 (DNN: Heart Disease)
**Objective:** Build a Deep Neural Network (DNN) model to predict heart disease using the **UCI Heart Disease dataset**
The workflow includes **data preprocessing**, **splitting**, **model development**, **tuning**, **evaluation**, and **performance analysis**.
### Dataset Overview & Source
This notebook uses the Kaggle Heart Disease dataset as provided in `heart.csv`, containing **1,025 records** with 14 clinical attributes.  
The target `target` indicates heart disease presence (`1`) or absence (`0`).
The dataset is publicly available at [Kaggle – Heart Disease Dataset](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset?select=heart.csv)


## Installing the Necessary Libraries


In [ ]:
# Importing Libraries and Setting Random Seeds
import pandas as pd
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras.layers import BatchNormalization
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
seed_value = 42
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)
tf.__version__


## Download and Explore Data
Load the **heart.csv** dataset and display the first few rows to verify successful loading and table structure.


In [ ]:
# Load the dataset
file_path = '/content/heart.csv'
df = pd.read_csv(file_path)
# Display the first 5 rows
df.head()


### Check Data Structure and Missing Values
Display dataset information using `df.info()`
This helps identify any missing or inconsistent data before preprocessing.


In [ ]:
# Check dataset structure and missing values
df.info()


### Statistical Data Summary
Use `df.describe()` to generate descriptive statistics for all numeric columns.


In [ ]:
# Display statistical summary for numerical features
df.describe()


### Verify Target Column and Class Balance
Confirm that the correct target column (`target`) exists for classification.  
Then, check the distribution of both classes (0 = no disease, 1 = disease) to ensure balanced data.


In [ ]:
# Verify target column
possible_targets = [c for c in df.columns if c.lower() in ['target', 'label', 'class', 'outcome']]
assert len(possible_targets) >= 1, "Please specify the correct target column, e.g., 'target'."
TARGET = possible_targets[0]
print("TARGET =", TARGET, "| unique:", df[TARGET].unique())
# Check class balance
print("Class balance:")
print(df[TARGET].value_counts(normalize=True).round(3))


## Data Preprocessing & Data Splitting
This section covers:
- Separating features and the target variable
- Defining categorical and numerical columns
- Splitting the dataset into 70% training, 10% validation, and 20% testing
- Applying **One-Hot Encoding** after splitting to prevent data leakage
- Aligning columns across all sets to match the training data
- Scaling numerical features using **StandardScaler** for faster model convergence


In [ ]:
# Separate features and target from the raw dataframe
feature_cols = [c for c in df.columns if c.lower() != 'target']
X_raw = df[feature_cols].copy()
y = df['target'].copy()
# Define categorical and numerical columns
categorical_cols = ['cp', 'restecg', 'slope', 'ca', 'thal']
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
# Split 80/20
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y
)
# Extract validation set
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.125, random_state=42, stratify=y_train
)
# One-Hot Encoding after splitting (to avoid leakage)
X_train = pd.get_dummies(X_train_raw, columns=categorical_cols, drop_first=False)
X_val = pd.get_dummies(X_val_raw, columns=categorical_cols, drop_first=False)
X_test = pd.get_dummies(X_test_raw, columns=categorical_cols, drop_first=False)
# Align columns in validation and test sets with training set
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
# StandardScaler: fit on training, transform on validation/test
scaler = StandardScaler()
X_train.loc[:, numerical_cols] = scaler.fit_transform(X_train.loc[:, numerical_cols])
X_val.loc[:, numerical_cols] = scaler.transform(X_val.loc[:, numerical_cols])
X_test.loc[:, numerical_cols] = scaler.transform(X_test.loc[:, numerical_cols])
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
X_train.head()


## Model Development
Build a deep neural network (DNN) using **Keras**.  
The model includes an input layer, one or more hidden layers with `ReLU` activation, and a single output layer with `Sigmoid` activation for binary classification.  
The **Adam** optimizer is used with a small learning rate to ensure stable convergence.


In [ ]:
# Model creation function
def create_model(n_hidden_layers=1, n_neurons=64, dropout_rate=0.3, learning_rate=0.001):
    model = Sequential()
    # Input layer
    model.add(Dense(n_neurons, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dropout(dropout_rate))
    # Hidden layers
    for _ in range(n_hidden_layers):
        model.add(Dense(n_neurons, activation='relu'))
        model.add(Dropout(dropout_rate))
    # Output layer
    model.add(Dense(1, activation='sigmoid'))
    # Compile model
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


## Early Stopping Configuration
Use **EarlyStopping** to monitor `val_accuracy` during training.  
Training stops if validation accuracy does not improve after 10 epochs, preventing overfitting and restoring the best model weights.


In [ ]:
# EarlyStopping callback
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    verbose=1,
    restore_best_weights=True
)


## First Experiment – Baseline Model
Build an initial baseline model to establish reference performance.  
**Parameters:** `layers=1`, `neurons=64`, `dropout=0.4`, `lr=0.001`, `batch_size=64`  
Training stopped early at epoch 14 with **accuracy ≈ 0.83** on the test set, serving as a solid starting point for later improvements.


In [ ]:
# First Experiment – Baseline Model
print("Building Baseline Model...")
baseline_model = create_model(
    n_hidden_layers=1,
    n_neurons=64,
    dropout_rate=0.4,
    learning_rate=0.001
)
baseline_model.fit(
    X_train, y_train,
    epochs=100, batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=0
)
loss, accuracy = baseline_model.evaluate(X_test, y_test, verbose=0)
print(f"\nResult: Accuracy = {accuracy:.4f}")


## Second Experiment – Smaller Batch Size
In this experiment, the batch size was reduced from 64 to 32 to allow more frequent weight updates and finer optimization.  
**Parameters:** `layers=1`, `neurons=64`, `dropout=0.4`, `lr=0.001`, `batch_size=32`  
This adjustment improved test accuracy to **≈ 0.94**, showing better training stability and generalization.


In [ ]:
# Second Experiment – Smaller Batch Size
print("\nImproving Model...")
improved_model = create_model(
    n_hidden_layers=1,
    n_neurons=64,
    dropout_rate=0.4,
    learning_rate=0.001
)
improved_model.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=0
)
loss, accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print(f"\nResult: Accuracy = {accuracy:.4f}")


## Third Experiment – Reduced Dropout Rate
This step reduces the dropout rate from `0.4` to `0.2`, allowing the model to learn more complex patterns.  
**Parameters:** `layers=1`, `neurons=64`, `dropout=0.2`, `lr=0.001`, `batch_size=32`  
As a result, the model achieved **≈ 0.97 accuracy** on the test set — the best performance among all experiments.


In [ ]:
# Third Experiment – Best Model
print("\nBest Model")
best_model = create_model(
    n_hidden_layers=1,
    n_neurons=64,
    dropout_rate=0.2,
    learning_rate=0.001
)
best_model.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=0
)
loss, accuracy = best_model.evaluate(X_test, y_test, verbose=0)
print(f"\nBest Model Result: Accuracy = {accuracy:.4f}")


To further explore the model’s performance, two additional experiments were conducted using an enhanced model builder (create_model_v2) that supports L2 regularization and Batch Normalization.
These experiments aim to test the effect of network depth, regularization, and learning rate adjustments on the model’s generalization and stability.


In [ ]:
def create_model_v2(n_hidden_layers=1, n_neurons=64, dropout_rate=0.3, learning_rate=1e-3,
                    l2=0.0, use_bn=False):
    model = Sequential()
    model.add(Dense(
        n_neurons, activation='relu', input_shape=(X_train.shape[1],),
        kernel_regularizer=regularizers.l2(l2) if l2>0 else None
    ))
    if use_bn: model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))
    for _ in range(n_hidden_layers):
        model.add(Dense(
            n_neurons, activation='relu',
            kernel_regularizer=regularizers.l2(l2) if l2>0 else None
        ))
        if use_bn: model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
model_2layers = create_model_v2(
    n_hidden_layers=2, n_neurons=32, dropout_rate=0.3, learning_rate=1e-3,
    l2=1e-4, use_bn=False
)
model_2layers.fit(
    X_train, y_train, epochs=100, batch_size=32,
    validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=0
)
loss, acc = model_2layers.evaluate(X_test, y_test, verbose=0)
print(f"Result: Accuracy = {acc:.4f}")
runs.append({"name":"model_2layers","layers":2,"units":32,"dropout":0.3,"lr":1e-3,"batch":32,
             "epochs_trained": len(model_2layers.history.history['loss']),
             "test_accuracy": float(acc)})


In [ ]:
model_3layers_bn = create_model_v2(
    n_hidden_layers=3, n_neurons=32, dropout_rate=0.3, learning_rate=5e-4,
    l2=1e-4, use_bn=True
)
model_3layers_bn.fit(
    X_train, y_train, epochs=100, batch_size=32,
    validation_data=(X_val, y_val), callbacks=[early_stopping], verbose=0
)
loss, acc = model_3layers_bn.evaluate(X_test, y_test, verbose=0)
print(f"Result: Accuracy = {acc:.4f}")
runs.append({"name":"model_3layers_bn","layers":3,"units":32,"dropout":0.3,"lr":5e-4,"batch":32,
             "epochs_trained": len(model_3layers_bn.history.history['loss']),
             "test_accuracy": float(acc)})


### Model Building Summary
Three different model configurations were tested.  
The final model used:
- One hidden layer (64 neurons)  
- Low dropout rate (0.2)  
- Learning rate of 0.001  
- Batch size of 32  
This setup achieved the best balance between performance and training stability, with about **96.6% accuracy** on the test set.


## Model Evaluation
Evaluate the best-performing model on the test set.  
Generate predictions and compute the classification report, including **accuracy**, **precision**, **recall**, and **F1-score** for each class.
The model achieved balanced performance across both classes, with an overall accuracy of **≈ 96.6%**.


In [ ]:
# Generate predictions on the test data
y_pred_proba = best_model.predict(X_test, verbose=0).ravel()# Flatten predictions
y_pred = (y_pred_proba >= 0.5).astype(int)    # Convert to binary output
# Print classification report
print("Classification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=['No Disease (0)', 'Disease (1)'],
    digits=4,
    zero_division=0
))


### Confusion Matrix
The confusion matrix illustrates how the model classified each class.  
It shows the number of correct and incorrect predictions for both healthy and diseased patients.  
**Analysis:**  
- Correctly identified **94 healthy** and **104 diseased** patients.  
- Only **7 samples** were misclassified (6 from class 0, 1 from class 1).  
This indicates that the model has very high accuracy and strong generalization ability.


In [ ]:
# Compute and visualize the confusion matrix
print("\nConfusion Matrix")
cm = confusion_matrix(y_test, y_pred)
# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Disease (0)', 'Disease (1)'],
    yticklabels=['No Disease (0)', 'Disease (1)']
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


### Misclassified Samples Analysis
To better understand model behavior, we inspect some of the samples that were misclassified.  
This helps identify **borderline cases** where features overlap between the two classes.  
**Observations:**  
Most misclassified patients fall into borderline scenarios such as:  
- High blood pressure or cholesterol without other strong indicators.  
- Older patients with no clear cardiac symptoms.  
Such cases suggest that additional medical data (e.g., history or lab results) could improve model accuracy in the future.


In [ ]:
# Convert y_test to a numpy array for easier comparison
y_test_np = y_test.to_numpy()
# Identify misclassified samples
misclassified_indices = np.where(y_pred != y_test_np)[0]
# Display some misclassified samples
if len(misclassified_indices) > 0:
    print(f"\nFound {len(misclassified_indices)} misclassified samples. Here are a few:")
    # Restore original numeric values
    X_test_original = X_test.copy()
    X_test_original[numerical_cols] = scaler.inverse_transform(X_test[numerical_cols])
    # Show first 5 misclassified samples
    for index in misclassified_indices[:5]:
        print(f"Sample Index: {index}")
        print("Original Patient Data:")
        print(X_test_original.iloc[index])
        print(f"\nActual Label: {y_test.iloc[index]}")
        print(f"Predicted Label: {int(y_pred[index])}")
else:
    print("\nNo misclassified samples found.")


# hyperparameters


In [ ]:
# consolidated hyperparameters summary
runs = []
def safe_eval(model, name, layers, units, dropout, lr, batch):
    if name in globals() and globals()[name] is not None:
        m = globals()[name]
        try:
            epochs_trained = len(m.history.history['loss'])
        except Exception:
            epochs_trained = None
        loss, acc = m.evaluate(X_test, y_test, verbose=0)
        runs.append({
            "name": name,
            "layers": layers,
            "units": units,
            "dropout": dropout,
            "lr": lr,
            "batch": batch,
            "epochs_trained": epochs_trained,
            "test_accuracy": float(acc)
        })
# Experience evaluation
safe_eval(baseline_model, "baseline_model", layers=1, units=64, dropout=0.4, lr=0.001, batch=64)
safe_eval(improved_model, "improved_model", layers=1, units=64, dropout=0.4, lr=0.001, batch=32)
safe_eval(best_model,     "best_model",     layers=1, units=64, dropout=0.2, lr=0.001, batch=32)
import pandas as pd
summary_df = pd.DataFrame(runs).sort_values("test_accuracy", ascending=False)
print("\n Best Hyperparameters (by test accuracy) ")
if len(summary_df):
    best = summary_df.iloc[0]
    print(
        f"name={best['name']}, layers={best['layers']}, units={best['units']}, "
        f"dropout={best['dropout']}, lr={best['lr']}, batch={best['batch']}, "
        f"epochs_trained={best['epochs_trained']}, test_acc={best['test_accuracy']:.4f}"
    )
else:
    print("No trained models found.")
summary_df


## Final Model Evaluation Summary
The model demonstrated **excellent performance** across all evaluation metrics, achieving an overall accuracy of approximately **96.6%**.  
Misclassifications were minimal and primarily occurred in **borderline cases** with overlapping clinical features.


## Extra Insights – Model Training Analysis
In this section, we analyze the **training curves** to evaluate model performance over time.  
We examine the `accuracy` and `loss` trends from the training history to confirm model stability and learning behavior.
### Accuracy & Loss Curves Analysis
- The **accuracy curves** for both training and validation rise steadily and converge, indicating stable learning.  
- The **loss curves** show consistent decrease with close convergence at the end.  
- These trends confirm that the model **learned effectively** without signs of overfitting or underfitting.  
- The small gap between training and validation performance reflects **strong generalization** on unseen data.


In [ ]:
# Training history analysis
hist = best_model.history
plt.figure(figsize=(6,4))
plt.plot(hist.history['accuracy'])
plt.plot(hist.history['val_accuracy'])
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Over Epochs')
plt.legend(['Train', 'Validation'])
plt.show()
plt.figure(figsize=(6,4))
plt.plot(hist.history['loss'])
plt.plot(hist.history['val_loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Over Epochs')
plt.legend(['Train', 'Validation'])
plt.show()


## ROC Curve & AUC
To further evaluate model performance, we use the **ROC Curve**, which illustrates the relationship between:  
- **True Positive Rate .  
- **False Positive Rate .  
We also compute the **AUC **.
### Results Analysis
- The obtained **AUC = 0.9813**, indicating an **excellent discrimination ability** between healthy and diseased patients.  
- The ROC curve is very close to the top-left corner, reflecting an **almost perfect classifier**.  
- These results confirm the model’s **robustness and reliability** for real-world heart disease prediction.


In [ ]:
# ROC Curve & AUC
from sklearn.metrics import roc_auc_score, roc_curve
y_prob = best_model.predict(X_test, verbose=0).ravel()
auc = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)
print(f"AUC = {auc:.4f}")
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.show()


##Summary & Conclusion
The deep neural network (DNN) built with **Keras** demonstrated outstanding performance in predicting heart disease, achieving **97% accuracy** and an **AUC of 0.98**.
**Key Highlights:**
- Simple yet effective architecture with Dense and Dropout layers.  
- EarlyStopping prevented overfitting and ensured stable training.  
- Strong results across validation and test sets with minimal misclassifications.  
**Future Improvements:**
- Experiment with deeper or wider architectures for potential gains.  
- Test on data from different medical sources to improve generalization.  
- Apply model explainability techniques to understand key predictive factors.
